
# 01 &middot; Exploratory data analysis

**~40 minutes. Strongly recommended.**

You cannot decide what to model until you know what you are looking at. This
notebook is where you form the opinions you will spend the rest of the day
testing.

Four questions:

1. What is actually measured, and how much of it is missing?
2. Why does everyone log-transform this data, and did they do it right?
3. Which endpoints look related to each other?
4. Are the test molecules anything like the training molecules?

Question 4 is the one most people skip and most people should not.

In [ ]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec matplotlib seaborn

# Get common.py. If you uploaded it yourself (folder icon in the left sidebar),
# this leaves your copy alone -- it only downloads when the file is missing.
!test -s common.py || wget -q -O common.py https://raw.githubusercontent.com/CHANGE-ME/admet-hackathon/main/common.py

import os, sys
assert os.path.exists("common.py") and os.path.getsize("common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)   # force a fresh read if you just re-uploaded it
import common
common.setup(pair="CHANGE-ME")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style("whitegrid"); sns.set_context("notebook")

train = common.load_train()
test  = common.load_test()
train_raw = common.load_train(log_scale=False)
print(len(train), "train /", len(test), "test")

---
## 1. Sparsity

Each row is a molecule, each column an assay. Yellow means measured.

In [ ]:
m = train[common.ENDPOINTS].notna()
order = m.sum(axis=1).sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(7, 8))
ax.imshow(m.loc[order].to_numpy(), aspect="auto", interpolation="nearest",
          cmap="viridis")
ax.set_xticks(range(len(common.ENDPOINTS)))
ax.set_xticklabels(common.ENDPOINTS, rotation=90)
ax.set_ylabel("molecules (sorted by how much data they have)")
ax.set_title("What was actually measured")
plt.tight_layout(); plt.show()

print("assays measured per molecule:")
print(m.sum(axis=1).value_counts().sort_index().to_string())

**Look at the shape of that.** A small number of compounds got the
full panel; most got one or two assays. The compounds with everything measured
are the ones the project cared about &mdash; which means the missingness is
*not random*. It correlates with how promising a compound looked at the time.

Two consequences worth holding onto:

- Dropping incomplete rows throws away most of your data.
- A model trained only on fully-measured compounds is trained on an unusual,
  pre-selected subset.

---
## 2. The log transform

Every endpoint except LogD gets log-transformed before modelling. Let us check
that this is a good idea rather than a ritual.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
raw_cols = [c for c in common.RAW_ENDPOINTS if c != "LogD"]
for ax, col in zip(axes.ravel(), raw_cols):
    v = pd.to_numeric(train_raw[col], errors="coerce").dropna()
    ax.hist(v, bins=50, color="#c44")
    ax.set_title(col, fontsize=9); ax.set_yticks([])
fig.suptitle("RAW units -- note the long right tails", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), common.ENDPOINTS):
    v = train[col].dropna()
    ax.hist(v, bins=50, color="#468")
    ax.set_title(f"{col}  (n={len(v):,})", fontsize=9); ax.set_yticks([])
fig.suptitle("LOG scale -- what you actually model", y=1.01)
plt.tight_layout(); plt.show()

### &#9654;&#65039; Predict first

**Look at the log-scale histograms. Which endpoint's distribution is going to cause a model the most trouble, and why?**

*You are looking for a shape that a regression model handles badly. Think about what the model does when the data is piled up at both ends.*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

### Did they transform it correctly?

Here is the transform, copied out of the official OpenADMET tutorial:

```python
value = value + 1                      # avoid log(0)
value = np.log10(value * multiplier)   # unit conversion
```

Implement it yourself and check it reproduces what `common.load_train()` gave
you.

In [ ]:
# YOUR CODE: reproduce the log-scale KSOL column from the raw one.
# KSOL is in micromolar, so the multiplier is 1e-6.

mine = ...   # <-- replace this

# check
ref = train["LogS"]
# print("max difference:", np.nanmax(np.abs(mine - ref)))

In [ ]:
#@title Reveal { display-mode: "form" }
mine = np.log10((pd.to_numeric(train_raw["KSOL"], errors="coerce") + 1) * 1e-6)
print("max difference:", np.nanmax(np.abs(mine - train["LogS"])))

### Now argue with it

That `+1` is a zero-guard: solubility can be measured as 0, and `log10(0)` is
undefined. Fine. But look at what it actually does.

- For **KSOL**, you add 1 &micro;M to every measurement, then convert to molar.
  A compound measured at 1 &micro;M becomes 2 &micro;M. A compound at 500 &micro;M
  is essentially unchanged. So the guard distorts exactly the low-solubility
  compounds you most want to identify.
- For **HLM CLint**, the multiplier is 1, so you are adding 1 unit in the
  assay's own units.
- The size of the nudge relative to the data is therefore **different for every
  endpoint**, because the units are different.

Is that a problem? Maybe not much. But it is a choice someone made, it is
inherited by everyone who uses this tutorial, and nobody wrote down why. Worth
knowing it is there.

*Optional, for anyone who wants a fight:* try `log10(x + eps)` with a small
epsilon, or a signed log, and see whether your KSOL model improves.

---
## 3. Which endpoints move together?

If two endpoints are measuring related physics, a model that learns them
*together* can use one to help the other. This plot is the starting point for
task grouping in `card_multitask`.

In [ ]:
corr = train[common.ENDPOINTS].corr(method="spearman", min_periods=30)
overlap = train[common.ENDPOINTS].notna().astype(int).T @ train[common.ENDPOINTS].notna().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=axes[0], cbar=False)
axes[0].set_title("Spearman correlation (co-measured molecules only)")
sns.heatmap(overlap, annot=True, fmt="d", cmap="Greys", ax=axes[1], cbar=False)
axes[1].set_title("How many molecules have BOTH endpoints")
plt.tight_layout(); plt.show()

**Read the two panels together.** A correlation of 0.7 computed on
25 molecules is not evidence of anything. Sparse data means some of these
correlations are extremely noisy, and the right-hand panel tells you which
ones to ignore.

Also &mdash; and this trips people up &mdash; **label correlation is not the same
as task affinity**. Two endpoints can help each other in a multitask model
because they need the same *internal representation* of a molecule, even if
their measured values are uncorrelated. Correlation is a hint, not an answer.

---
## 4. Chemical space: is this interpolation or extrapolation?

This is the question that decides whether any of the modelling that follows
will work.

For each test molecule, find its most similar training molecule (Tanimoto
similarity on Morgan fingerprints). High similarity means you are
**interpolating** inside known chemistry. Low similarity means you are
**extrapolating**, and models are much worse at that.

### &#9654;&#65039; Predict first

**The test set is late-stage compounds from the same discovery campaign as the training set. Do you expect their nearest-neighbour similarity to be high or low? Guess a number between 0 and 1.**

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

In [ ]:
fp_train = common.morgan_fingerprints(train[common.SMILES_COL])
fp_test  = common.morgan_fingerprints(test[common.SMILES_COL])

nn_test  = common.nearest_neighbour_similarity(fp_test, fp_train)
nn_train = common.nearest_neighbour_similarity(fp_train, fp_train, exclude_self=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(nn_train, bins=50, alpha=.6, label="train vs train", density=True)
ax.hist(nn_test,  bins=50, alpha=.6, label="test vs train",  density=True)
ax.axvline(common.MORGAN2_NOISE_FLOOR, color="k", ls="--",
           label=f"noise floor ({common.MORGAN2_NOISE_FLOOR})")
ax.set_xlabel("Tanimoto similarity to nearest training molecule")
ax.legend(); ax.set_title("Chemical space overlap")
plt.tight_layout(); plt.show()

print(f"median test->train similarity: {np.nanmedian(nn_test):.3f}")
print(f"fraction below the noise floor: {np.nanmean(nn_test < common.MORGAN2_NOISE_FLOOR):.1%}")

### Why the dashed line matters

For 2048-bit Morgan-2 fingerprints, a similarity of about **0.27** is the
level you get comparing two *random* molecules. Below that, the phrase
"nearest neighbour" is meaningless &mdash; the closest thing in your training set
carries no more information about your query than a molecule picked at random.

Hold that number. It is the punchline of the whole day.

When OpenADMET took the public web tools **ADMET-AI** and **ADMETlab 3.0** and
ran them zero-shot on this exact test set, they got **negative R&sup2;** on
several endpoints &mdash; worse than guessing. The reason was not bad
architecture. Both use the same Chemprop-style GNNs the challenge winners
used. The reason was that the nearest-neighbour similarity from this campaign
to their training data (ChEMBL, TDC) sits **below 0.3**.

Same test molecules. Same kind of model. The only thing that changed was
whether the training data lived in the same chemical space. That is the whole
ballgame.

---
## 5. Endpoint difficulty: form a hypothesis

Before you model anything, write down which endpoints you think will be hard.
This table gives you the raw material.

In [ ]:
rows = []
for e in common.ENDPOINTS:
    v = train[e].dropna()
    rows.append({"endpoint": e, "n": len(v), "std": v.std(),
                 "range": v.max() - v.min(), "skew": v.skew()})
summary = pd.DataFrame(rows).set_index("endpoint").round(2)
summary

Some things to weigh, which are chemistry rather than statistics:

- **LogD is additive.** Fragment contributions roughly sum, and they transfer
  between molecules. Additive properties are learnable.
- **Solubility is not additive.** It comes out of crystal packing and
  solvation, which do not decompose into fragment contributions.
- **Intrinsic clearance has activity cliffs.** One methyl group in the wrong
  place changes which metabolic soft spot the enzyme attacks. Models
  generalise badly across cliffs.
- **Efflux is active transport.** P-gp recognition depends on 3D pharmacophore
  complementarity, not on additive bulk properties.
- **Protein binding tracks lipophilicity and size**, so it inherits LogD's
  learnability &mdash; and it has a narrow dynamic range, which flatters any
  metric you compute on it.

Note that a small `std` makes an endpoint *look* easy: if everything is close
to the mean, guessing near the mean is a decent strategy. Keep that in mind
when you interpret your scores later.

---
## Take to the planning session

Write down three things:

1. Which endpoint you expect to be hardest, **and the chemical reason**.
2. Which endpoints you would group together in a multitask model.
3. Whether you think external public data will help here &mdash; given what you
   just saw about chemical space overlap.

Number 3 is a real bet and reasonable people disagreed about it during the
actual challenge.